# 🎬 StudioCut — Notebook VTT Parser (`clip-cli`)

This Google Colab notebook receives a WebVTT (`.vtt`) subtitle/caption file, cleans the text (removing VTT headers, timestamps, HTML/voice tags, and deduplicating repetitive subtitle lines), displays manual verification output, and stores the resulting cleaned transcript into **Google Drive** under the `clip-cli` directory.

## 📁 Step 1: Mount Google Drive

Mount Google Drive to enable reading/writing directly from your Drive storage inside the `clip-cli` directory.

In [ ]:
#@title Mount Google Drive { display-mode: "form" }
import os

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print("✅ Google Drive successfully mounted.")
except ImportError:
    print("ℹ️ Running outside Google Colab environment. Operating in local mode.")

## ⚙️ Step 2: Configuration & Parameters

Configure input/output parameters using native Colab form controls.

In [ ]:
#@title Configuration Parameters { display-mode: "form" }
vtt_file_path = "sample.vtt" #@param {type:"string"}
drive_output_folder = "/content/drive/MyDrive/clip-cli" #@param {type:"string"}
output_filename = "cleaned_transcript.txt" #@param {type:"string"}
preview_cues_count = 5 #@param {type:"integer"}

print(f"📌 VTT File Path: {vtt_file_path}")
print(f"📌 Drive Output Folder: {drive_output_folder}")
print(f"📌 Output Filename: {output_filename}")

## 📥 Step 3: Input VTT File Ingestion

Load the VTT file from the specified path, or trigger Colab file upload if the specified file is not found.

In [ ]:
import os

vtt_content = ""
target_path = vtt_file_path.strip()

if os.path.exists(target_path):
    with open(target_path, 'r', encoding='utf-8') as f:
        vtt_content = f.read()
    print(f"✅ Successfully loaded '{target_path}' ({len(vtt_content)} chars).")
else:
    print(f"⚠️ File '{target_path}' not found.")
    if IS_COLAB:
        from google.colab import files
        print("📤 Please upload a .vtt file:")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            if fname.lower().endswith('.vtt'):
                vtt_content = data.decode('utf-8', errors='ignore')
                vtt_file_path = fname
                print(f"✅ Loaded uploaded file: {fname}")
                break
    else:
        raise FileNotFoundError(f"Cannot find VTT file at path: {target_path}")

## 🧹 Step 4: Native VTT Parser & Text Cleaning Algorithm

Parse timestamps, strip VTT header lines (`WEBVTT`, `NOTE`, `STYLE`), clean HTML/voice tags, and remove consecutive duplicate phrases.

In [ ]:
import re
import json

def clean_vtt_transcript(vtt_text):
    lines = vtt_text.splitlines()
    cues = []
    current_cue = None
    timestamp_pattern = re.compile(r'(\d{2}:)?\d{2}:\d{2}[\.,]\d{3}\s*-->\s*(\d{2}:)?\d{2}:\d{2}[\.,]\d{3}')

    for line in lines:
        line_str = line.strip()
        if not line_str or line_str.startswith('WEBVTT') or line_str.startswith('NOTE') or line_str.startswith('STYLE'):
            continue
        
        if timestamp_pattern.search(line_str):
            parts = line_str.split('-->')
            start = parts[0].strip()
            end = parts[1].strip().split()[0]
            current_cue = {'start': start, 'end': end, 'raw': [], 'clean': ''}
            cues.append(current_cue)
        elif current_cue is not None and not line_str.isdigit():
            current_cue['raw'].append(line_str)

    clean_segments = []
    prev_clean_text = ''

    for cue in cues:
        raw_text = ' '.join(cue['raw'])
        # Remove HTML and speaker tags, e.g. <v Host> or <b>
        clean_text = re.sub(r'<[^>]*>', '', raw_text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        cue['clean'] = clean_text
        
        # Deduplicate identical consecutive cue lines
        if clean_text and clean_text != prev_clean_text:
            clean_segments.append(clean_text)
            prev_clean_text = clean_text

    raw_combined = ' '.join([' '.join(c['raw']) for c in cues])
    cleaned_full_text = ' '.join(clean_segments)

    # Additional consecutive word deduplication for streaming auto-captions
    words = cleaned_full_text.split()
    dedup_words = []
    for w in words:
        if not dedup_words or dedup_words[-1].lower() != w.lower():
            dedup_words.append(w)
    final_cleaned_transcript = ' '.join(dedup_words)

    return {
        'cleaned_transcript': final_cleaned_transcript,
        'raw_combined': raw_combined,
        'cues': cues,
        'total_cues': len(cues)
    }

parsing_result = clean_vtt_transcript(vtt_content)
print(f"✅ Cleaning complete. Parsed {parsing_result['total_cues']} cues.")
print(f"📊 Cleaned text length: {len(parsing_result['cleaned_transcript'])} characters.")

## 🔍 Step 5: Manual Verification of Outcome

Output part of the result alongside the corresponding raw VTT text to enable side-by-side manual verification.

In [ ]:
from IPython.display import HTML, display

def display_manual_verification(result, limit=5):
    cues = result['cues'][:limit]
    
    html_content = """
    <h3>🔍 Side-by-Side Manual Verification (First """ + str(limit) + """ Cues)</h3>
    <table border="1" style="border-collapse: collapse; width: 100%; font-family: sans-serif;">
      <thead>
        <tr style="background-color: #f2f2f2; text-align: left;">
          <th style="padding: 8px; width: 15%;">Timestamp</th>
          <th style="padding: 8px; width: 42.5%;">Raw VTT Cue Text</th>
          <th style="padding: 8px; width: 42.5%;">Cleaned Text</th>
        </tr>
      </thead>
      <tbody>
    """
    
    for i, c in enumerate(cues):
        raw_str = ' '.join(c['raw'])
        bg_color = "#ffffff" if i % 2 == 0 else "#f9f9f9"
        html_content += f"""
        <tr style="background-color: {bg_color};">
          <td style="padding: 8px; font-size: 12px; font-family: monospace;">{c['start']} &rarr; {c['end']}</td>
          <td style="padding: 8px; font-size: 13px; color: #cc0000; font-family: monospace;">{raw_str}</td>
          <td style="padding: 8px; font-size: 13px; color: #008800; font-weight: bold;">{c['clean']}</td>
        </tr>
        """
        
    html_content += """
      </tbody>
    </table>
    """
    display(HTML(html_content))

display_manual_verification(parsing_result, limit=preview_cues_count)

print("\n--- 📜 CLEANED TRANSCRIPT FULL PREVIEW ---")
print(parsing_result['cleaned_transcript'])
print("------------------------------------------\n")

## 💾 Step 6: Export Output to Google Drive (`clip-cli`)

Save the cleaned transcript text (`.txt`) and parsed JSON metadata (`.json`) into Google Drive inside the `clip-cli` folder.

In [ ]:
import os
import json

# Determine target directory
if IS_COLAB:
    target_dir = drive_output_folder
else:
    target_dir = "./output/clip-cli"

os.makedirs(target_dir, exist_ok=True)

# Write cleaned transcript text file
output_txt_path = os.path.join(target_dir, output_filename)
with open(output_txt_path, 'w', encoding='utf-8') as f:
    f.write(parsing_result['cleaned_transcript'])

# Write detailed structured JSON export
json_filename = os.path.splitext(output_filename)[0] + ".json"
output_json_path = os.path.join(target_dir, json_filename)
export_data = {
    "vtt_source": vtt_file_path,
    "total_cues": parsing_result['total_cues'],
    "cleaned_transcript": parsing_result['cleaned_transcript'],
    "cues": parsing_result['cues']
}
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2)

print(f"✅ Cleaned text file saved to: {output_txt_path}")
print(f"✅ Structured JSON file saved to: {output_json_path}")